# ISL Translator — Training Pipeline (digits 1-9 + letters A-Z, ~900 imgs/class)
**Dataset:** 35 classes, 900 images/class (V has 990) = 31,590 total images

### Status check (from the log after the utils.py fix)
The `utils.py` fix from the previous round worked as intended: validation samples
now land in sane, roughly-even per-class buckets (~150-210 each out of 6,318),
instead of all piling onto 7 classes. The `WARNING: class folder image counts are
imbalanced` message is just V having 90 extra images (990 vs 900) — `class_weight="balanced"`
already compensates for that automatically; it is **not** a meaningful contributor
to accuracy problems and needs no code change.

The pasted log stops right after model-building (before any epoch/accuracy output),
so there's no new accuracy result to diagnose yet from this run. This round's
changes are proactive tuning based on the dataset stats now confirmed correct,
plus tooling to diagnose whatever the next full run reports:

### Changes in this revision
| Issue | Root cause | Fix |
|---|---|---|
| `label_smoothing` silently had no effect (`[Warning] Installed TensorFlow does not support label_smoothing with SparseCategoricalCrossentropy`) | That constructor argument isn't available on all installed Keras versions, so it fell back to plain cross-entropy every time | `word_model.py` now uses a small `Loss` subclass that one-hots labels and calls `categorical_crossentropy(..., label_smoothing=...)`, which has supported smoothing for a long time — guaranteed to work and still serializes cleanly via `get_config()` |
| No visibility into *which* classes get confused with each other | Per-class accuracy alone doesn't show what a wrong prediction became | `train.ipynb` now prints a confusion-matrix-derived "top confused pairs" report after training — critical for a 35-class digits+letters model, since ISL has some numeral/letter signs with genuinely near-identical handshapes; that's a data/domain limit no amount of hyperparameter tuning fixes, and this report is how you tell "model needs more tuning" apart from "these two classes are inherently ambiguous in a single static frame" |
| Augmentation possibly too aggressive for fine-grained handshape differences | ~±20° rotation / 10% zoom can push already-similar classes closer together instead of just adding robustness | Rotation/zoom/translation dialed back (~±12.6° rotation, 6% zoom); brightness/contrast jitter (shape-preserving) kept |
| Fine-tuning depth was conservative for the dataset size | 30 unfrozen top layers was tuned for a smaller set; 31.5k balanced images can support more | `UNFREEZE_N` raised 30 -> 45, `unfreeze_top_layers()` now also takes `num_classes` (needed to rebuild the label-smoothing loss on recompile) |

Run this **after** replacing `utils.py` (from the previous round, already fixing the
label/shuffle desync) and `word_model.py` (this round) — retrain from scratch and
share the full log this time, including the per-class accuracy table and the new
top-confused-pairs section, so any remaining weak classes can be diagnosed precisely.

In [1]:
"""
train.ipynb - ISL Translator Training Pipeline
Digits 1-9 + Letters A-Z: 35 classes x 900 images/class (31,500 total)
"""

import os
import sys
sys.path.insert(0, r"C:\\Users\\kkani\\Documents\\py files\\ISL")

import numpy as np
import tensorflow as tf
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import importlib
import utils
importlib.reload(utils)

from word_model import build_model, unfreeze_top_layers, print_model_summary
from utils import (
    get_generators_from_directory,
    save_label_map, plot_training_history,
    MODEL_SAVE_PATH, IMG_SIZE
)

# -----------------------------------------------------------------------------
# PATHS
# -----------------------------------------------------------------------------
DATASET_PATH     = r"C:\\Users\\kkani\\Downloads\\archive1"
BASE_DIR         = "C:\\Users\\kkani\\Documents\\py files\\ISL"
PHASE1_SAVE_PATH = os.path.join(BASE_DIR, "word\\isl_best_model.keras")

# -----------------------------------------------------------------------------
# HYPER-PARAMETERS
# -----------------------------------------------------------------------------
BATCH_SIZE = 64

# Phase 1 - train classification head only (base frozen)
PHASE1_EPOCHS = 20
PHASE1_LR     = 1e-3

# Phase 2 - fine-tune top N layers of the base model
PHASE2_EPOCHS  = 15
FINE_TUNE_LR   = 5e-5
UNFREEZE_N     = 45

# Skip Phase 2 when Phase 1 accuracy is below this floor
PHASE2_MIN_ACC = 0.40

# Label smoothing passed to build_model()/unfreeze_top_layers() in word_model.py
LABEL_SMOOTHING = 0.05


# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------

def configure_cpu():
    import multiprocessing
    n = multiprocessing.cpu_count()
    tf.config.threading.set_intra_op_parallelism_threads(n)
    tf.config.threading.set_inter_op_parallelism_threads(n)
    print(f"[Train] CPU mode - {n} cores.")


def compute_class_weights(train_labels, num_classes):
    import numpy as np
    from sklearn.utils.class_weight import compute_class_weight

    weights = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(num_classes),
        y=train_labels
    )
    return {i: w for i, w in enumerate(weights)}


def phase1_callbacks():
    os.makedirs(os.path.dirname(PHASE1_SAVE_PATH) or ".", exist_ok=True)
    return [
        ModelCheckpoint(
            filepath=PHASE1_SAVE_PATH,
            monitor="val_accuracy",
            save_best_only=True,
            verbose=1,
            mode="max"
        ),
        EarlyStopping(
            monitor="val_accuracy",
            patience=5,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.4,
            patience=3,
            min_lr=1e-7,
            verbose=1
        ),
    ]


def phase2_callbacks():
    os.makedirs(os.path.dirname(MODEL_SAVE_PATH) or ".", exist_ok=True)
    return [
        ModelCheckpoint(
            filepath=MODEL_SAVE_PATH,
            monitor="val_accuracy",
            save_best_only=True,
            verbose=1,
            mode="max"
        ),
        EarlyStopping(
            monitor="val_accuracy",
            patience=5,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=3,
            min_lr=1e-8,
            verbose=1
        ),
    ]


def get_best_val_accuracy(history) -> float:
    accs = history.history.get("val_accuracy", [])
    return max(accs) if accs else 0.0


# -----------------------------------------------------------------------------
# Main
# -----------------------------------------------------------------------------

def main():
    if not os.path.isdir(DATASET_PATH):
        print(f"[ERROR] Dataset not found:\n  {DATASET_PATH}")
        sys.exit(1)

    configure_cpu()

    # -- Data generators ------------------------------------------------------
    print("\n[Train] Scanning dataset ...")
    train_gen, val_gen, label_map, class_names, num_classes, train_labels, val_labels = \
        get_generators_from_directory(DATASET_PATH, batch_size=BATCH_SIZE)

    print(f"[Train] Classes ({num_classes}): {class_names}")
    print(f"[Train] Batch size          : {BATCH_SIZE}")
    print(f"[Train] Steps/epoch (train) : {len(train_gen)}")
    print(f"[Train] Steps/epoch (val)   : {len(val_gen)}")

    if num_classes != 35:
        print(f"\n[WARN] Expected 35 classes (digits 1-9 + A-Z) but found "
              f"{num_classes}. Check DATASET_PATH folder structure against "
              f"label_map.json before trusting results.")

    print("\n[Train] Computing class weights ...")
    class_weights = compute_class_weights(train_labels, num_classes)

    # -- Build model -----------------------------------------------------------
    print("\n[Train] Building model ...")
    model, base_model = build_model(
        num_classes=num_classes,
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        learning_rate=PHASE1_LR,
        label_smoothing=LABEL_SMOOTHING
    )
    print_model_summary(model)

    print("\n[Train] == Preprocessing sanity check ==")
    print("  Generator rescale : 1/255  ->  images in [0, 1]")
    print("  Model preprocess  : NONE   (preprocess_input removed from word_model.py)")
    print("  Augmentation      : in-graph, training-only (rotation/zoom/translate/brightness/contrast, no flips)")
    print("  MobileNetV2 input : [0, 1] - consistent at train & inference\n")

    # =============================================================================
    # Phase 1 - Train classification head only (base FROZEN)
    # =============================================================================
    print("=" * 60)
    print(f"PHASE 1: Train head  (up to {PHASE1_EPOCHS} epochs, LR={PHASE1_LR})")
    print(f"         Best model -> {PHASE1_SAVE_PATH}")
    print("=" * 60)

    history1 = model.fit(
        train_gen,
        epochs=PHASE1_EPOCHS,
        validation_data=val_gen,
        class_weight=class_weights,
        callbacks=phase1_callbacks(),
        verbose=1
    )

    phase1_best_acc = get_best_val_accuracy(history1)
    print(f"\n[Train] Phase 1 best val_accuracy : {phase1_best_acc:.4f}")

    # =============================================================================
    # Phase 2 - Fine-tune top N layers of the base model
    # =============================================================================
    history2 = None

    if phase1_best_acc < PHASE2_MIN_ACC:
        print(f"\n[Train] Phase 1 accuracy ({phase1_best_acc:.1%}) below "
              f"{PHASE2_MIN_ACC:.0%} threshold - skipping Phase 2.")
        import shutil
        shutil.copy2(PHASE1_SAVE_PATH, MODEL_SAVE_PATH)
    else:
        print("\n" + "=" * 60)
        print(f"PHASE 2: Fine-tune top {UNFREEZE_N} base layers")
        print(f"         LR={FINE_TUNE_LR}  |  max {PHASE2_EPOCHS} epochs")
        print("         Loading Phase 1 best weights ...")
        print("=" * 60)

        model2, base_model2 = build_model(
            num_classes=num_classes,
            input_shape=(IMG_SIZE, IMG_SIZE, 3),
            learning_rate=PHASE1_LR,
            label_smoothing=LABEL_SMOOTHING
        )
        model2.load_weights(PHASE1_SAVE_PATH)

        model2 = unfreeze_top_layers(
            model2, base_model2,
            num_classes=num_classes,
            num_layers_to_unfreeze=UNFREEZE_N,
            learning_rate=FINE_TUNE_LR,
            label_smoothing=LABEL_SMOOTHING
        )

        history2 = model2.fit(
            train_gen,
            epochs=PHASE2_EPOCHS,
            validation_data=val_gen,
            class_weight=class_weights,
            callbacks=phase2_callbacks(),
            verbose=1
        )

        phase2_best_acc = get_best_val_accuracy(history2)
        print(f"\n[Train] Phase 2 best val_accuracy : {phase2_best_acc:.4f}")

        if phase2_best_acc < phase1_best_acc:
            print(f"[Train] Phase 2 ({phase2_best_acc:.1%}) worse than "
                  f"Phase 1 ({phase1_best_acc:.1%}) - reverting to Phase 1 model.")
            import shutil
            shutil.copy2(PHASE1_SAVE_PATH, MODEL_SAVE_PATH)
        else:
            print(f"[Train] Phase 2 improved accuracy "
                  f"({phase1_best_acc:.1%} -> {phase2_best_acc:.1%}).")

    # -- Merge histories & plot --------------------------------------------------
    class MergedHistory:
        def __init__(self, h1, h2):
            self.history = {
                k: h1.history[k] + (h2.history.get(k, []) if h2 else [])
                for k in h1.history
            }

    plot_training_history(MergedHistory(history1, history2))

    # -- Load final model for evaluation ------------------------------------------
    print(f"\n[Train] Loading final model from {MODEL_SAVE_PATH} for evaluation ...")
    final_model = tf.keras.models.load_model(MODEL_SAVE_PATH, compile=False)

    # -- Clean train-vs-val comparison (no augmentation, no dropout) -------------
    # The per-batch "accuracy"/"loss" printed during model.fit() is computed
    # with augmentation layers AND dropout ACTIVE (training=True forward pass),
    # while val_accuracy is computed with both OFF (training=False). That alone
    # can make training accuracy look far worse than validation accuracy even
    # when the model is learning fine — it's comparing "accuracy on hard,
    # noised-up batches" against "accuracy on clean images". This re-evaluates
    # both sets cleanly (model.evaluate() always runs in inference mode) so
    # they're an apples-to-apples comparison.
    final_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
                         metrics=["accuracy"])
    print("[Train] Clean (no-augmentation/no-dropout) accuracy check:")
    train_clean_loss, train_clean_acc = final_model.evaluate(train_gen, verbose=1)
    val_clean_loss, val_clean_acc = final_model.evaluate(val_gen, verbose=1)
    print(f"  Train (clean) : loss={train_clean_loss:.4f}  accuracy={train_clean_acc:.4f}")
    print(f"  Val   (clean) : loss={val_clean_loss:.4f}  accuracy={val_clean_acc:.4f}")
    gap = train_clean_acc - val_clean_acc
    if gap > 0.10:
        print(f"  [WARN] Train-val gap of {gap:.1%} once both are measured cleanly "
              f"suggests real overfitting - consider more augmentation/dropout, "
              f"or more data.")
    else:
        print(f"  [OK] Train/val gap is only {gap:.1%} once measured cleanly - "
              f"the raw training-time accuracy printed during fit() was misleading "
              f"due to augmentation + dropout being active, not a real problem.")

    # -- Per-class accuracy on validation set -------------------------------------
    print("\n[Train] Per-class accuracy on validation set:")
    preds        = final_model.predict(val_gen, verbose=1)
    pred_classes = np.argmax(preds, axis=1)
    true_classes = val_labels[:len(pred_classes)]

    print(f"\n  {'Letter':<10} {'Correct':>8} {'Total':>8} {'Accuracy':>10}")
    print("  " + "-" * 40)
    for i, name in enumerate(class_names):
        mask    = true_classes == i
        total   = mask.sum()
        correct = (pred_classes[mask] == i).sum()
        acc     = correct / total * 100 if total > 0 else 0
        flag    = "  <- LOW" if acc < 50 else ""
        print(f"  {name.upper():<10} {correct:>8} {total:>8} {acc:>9.1f}%{flag}")

    # -- Confusion matrix: find exactly which classes get mixed up ---------------
    # Per-class accuracy alone doesn't say *what* a wrong prediction became.
    # For a joint digits+letters ISL model, low accuracy on a specific class
    # is often explained by one or two other classes with a near-identical
    # handshape (a documented property of ISL, not something hyperparameter
    # tuning alone can fix) — this surfaces those pairs directly.
    from sklearn.metrics import confusion_matrix as _confusion_matrix

    cm = _confusion_matrix(true_classes, pred_classes, labels=list(range(num_classes)))
    print("\n[Train] Top confused class pairs (true -> predicted, count):")
    confusions = []
    for i in range(num_classes):
        for j in range(num_classes):
            if i != j and cm[i, j] > 0:
                confusions.append((cm[i, j], class_names[i], class_names[j]))
    confusions.sort(reverse=True)
    if confusions:
        for count, true_name, pred_name in confusions[:15]:
            print(f"    {true_name:>3} -> {pred_name:<3}  ({count} times)")
    else:
        print("    (no confusions - every validation sample was classified correctly)")

    overall = (pred_classes == true_classes).mean() * 100
    print(f"\n  Overall val accuracy : {overall:.2f}%")

    if overall < 70:
        print("\n  [HINT] Accuracy lower than expected for 900 imgs/class.")
        print("         Check data quality and class folder structure.")
    else:
        print("\n  [GOOD] Strong accuracy! Run python predict.py")

    print(f"\n  Final model -> {MODEL_SAVE_PATH}")
    print("  Run  python predict.py  to start live detection.\n")


if __name__ == "__main__":
    main()


[Train] CPU mode - 12 cores.

[Train] Scanning dataset ...
Found 42000 files belonging to 35 classes.
Using 33600 files for training.
Found 42000 files belonging to 35 classes.
Using 8400 files for validation.
[utils] All classes contain 1200 images.
[utils] Validation samples per class (sanity check):
    1        232
    2        271
    3        242
    4        238
    5        246
    6        234
    7        229
    8        245
    9        249
    A        246
    B        230
    C        260
    D        261
    E        239
    F        241
    G        239
    H        257
    I        239
    J        245
    K        211
    L        232
    M        215
    N        238
    O        256
    P        233
    Q        247
    R        244
    S        246
    T        225
    U        228
    V        242
    W        230
    X        265
    Y        218
    Z        227
[Train] Classes (35): ['1', '2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', 'C', 'D', 'E', 'F', 'G'

Model: "ISL_MobileNetV2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation                 │ (None, 224, 224, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom (RandomZoom)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_translation              │ (None, 224, 224, 3)    │             0 │
│ (RandomTranslation)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_brightness               │ (None, 224, 224, 3)    │             0 │
│ (RandomBrightness)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_contrast                 │ (None, 224, 224, 3)    │             0 │
│ (RandomContrast)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 384)            │       491,904 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 384)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        49,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 35)             │         4,515 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,808,803 (10.71 MB)

 Trainable params: 548,259 (2.09 MB)

 Non-trainable params: 2,260,544 (8.62 MB)


[Model] Trainable params     : 548,259
[Model] Non-trainable params : 2,260,544

[Train] == Preprocessing sanity check ==
  Generator rescale : 1/255  ->  images in [0, 1]
  Model preprocess  : NONE   (preprocess_input removed from word_model.py)
  Augmentation      : in-graph, training-only (rotation/zoom/translate/brightness/contrast, no flips)
  MobileNetV2 input : [0, 1] - consistent at train & inference

PHASE 1: Train head  (up to 20 epochs, LR=0.001)
         Best model -> C:\Users\kkani\Documents\py files\ISL\word\isl_best_model.keras
Epoch 1/20
525/525 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.1716 - loss: 3.2699
Epoch 1: val_accuracy improved from None to 0.07262, saving model to C:\Users\kkani\Documents\py files\ISL\word\isl_best_model.keras

Epoch 1: finished saving model to C:\Users\kkani\Documents\py files\ISL\word\isl_best_model.keras
525/525 ━━━━━━━━━━━━━━━━━━━━ 1228s 2s/step - accuracy: 0.2818 - loss: 2.8399 - val_accuracy: 0.0726 - val_loss: 3.4683 - learning_rat

c:\Users\kkani\Documents\py files\ISL\.conda\Lib\site-packages\keras\src\saving\saving_lib.py:801: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


525/525 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.4949 - loss: 2.1349
Epoch 1: val_accuracy improved from None to 0.89381, saving model to C:\Users\kkani\Documents\py files\ISL\word\isl_final_model.keras

Epoch 1: finished saving model to C:\Users\kkani\Documents\py files\ISL\word\isl_final_model.keras
525/525 ━━━━━━━━━━━━━━━━━━━━ 674s 1s/step - accuracy: 0.5043 - loss: 2.1021 - val_accuracy: 0.8938 - val_loss: 0.9126 - learning_rate: 5.0000e-05
Epoch 2/15
525/525 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5094 - loss: 2.0813
Epoch 2: val_accuracy improved from 0.89381 to 0.97143, saving model to C:\Users\kkani\Documents\py files\ISL\word\isl_final_model.keras

Epoch 2: finished saving model to C:\Users\kkani\Documents\py files\ISL\word\isl_final_model.keras
525/525 ━━━━━━━━━━━━━━━━━━━━ 666s 1s/step - accuracy: 0.5123 - loss: 2.0758 - val_accuracy: 0.9714 - val_loss: 0.6087 - learning_rate: 5.0000e-05
Epoch 3/15
525/525 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5104 - loss: 